# 3장 실습 — 최단경로 구현과 검증

2장에서 만든 자동차용 그래프와 출발·도착 노드를 입력으로 받아, 최소 통행시간과 경로를 구합니다.
이번 실습은 파일 두 개를 함께 엽니다.

- **이 노트북(`.ipynb`)**: 데이터를 준비하고 확인 셀을 실행합니다.
- **옆의 `ch03_dijkstra.py`**: `trace`와 `dijkstra` 함수를 작성하고 저장합니다.

`.py`에 함수를 작성·저장한 뒤 이 노트북의 해당 확인 셀을 다시 실행합니다.
노트북은 `import ch03_dijkstra as sol`로 파일을 불러오고, `sol.dijkstra(...)`로 함수를 호출합니다.
2장 노트북의 변수는 자동으로 넘어오지 않으므로 여기서 같은 데이터를 다시 읽습니다.

1~5절이 기본 실습입니다. 6절의 A*는 추가 실습이며 기본 채점에는 포함되지 않습니다.
미구현 상태에서도 노트북은 끝까지 실행되며, 확인 셀과 채점기에 미구현 안내가 나옵니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "smartmob").is_dir())
sys.path[:0] = [str(ROOT), str(ROOT / "labs")]

from lab import expect
import ch03_dijkstra as sol

print("함수를 작성할 파일:", sol.__file__)

위에 출력된 `.py` 파일을 편집합니다.
`autoreload`는 저장된 변경을 다음 셀 실행 전에 다시 불러옵니다.

## 1. 2장의 입력 다시 준비하기 (교재 3.1)

2장과 같은 자동차용 도로망, 하남시청·미사역 좌표를 사용합니다.

In [ ]:
from smartmob.data import load_road_graph

HANAM_CITY_HALL = (37.5393, 127.2148)
MISA_STATION = (37.5606, 127.1930)

drive = load_road_graph("hanam", modes=("drive",))
start = drive.nearest_node(*HANAM_CITY_HALL)
goal = drive.nearest_node(*MISA_STATION)

print(drive)
print("출발:", start, "도착:", goal)
print("출발 노드의 이웃:", drive.neighbors(start))

자동차 그래프는 노드 12,566개와 엣지 28,589개입니다.
이웃 목록의 각 항목은 `(이웃 노드, 구간 통행시간 초, 엣지 인덱스)`입니다.
이번에 작성할 함수는 이 목록을 읽어 목적지까지의 시간을 더합니다.

## 2. 작은 그래프에서 구현하기 (교재 3.2~3.3)

교재의 A~F를 코드에서는 `n1`~`n6`으로 부릅니다.
먼저 노드 표를 만듭니다.

In [ ]:
import pandas as pd
from smartmob.teaching.graph import RoadGraph

toy_nodes = pd.DataFrame([
    ("n1", 37.5000, 127.2000),  # A
    ("n2", 37.5030, 127.2000),  # B
    ("n3", 37.5080, 127.2000),  # C
    ("n4", 37.5050, 127.2010),  # D
    ("n5", 37.5500, 127.2500),  # E
    ("n6", 37.5520, 127.2500),  # F
], columns=["node_id", "lat", "lon"])

엣지 표의 두 번째 행은 A → C입니다.
모든 속도를 36km/h(10m/s)로 두므로 길이 600m가 60초, 2,000m가 200초가 됩니다.

In [ ]:
toy_edges = pd.DataFrame([
    ("e1_f_1_2", 600.0),   # A → B: 60초
    ("e2_f_1_3", 2000.0),  # A → C: 200초
    ("e3_f_2_3", 900.0),   # B → C: 90초
    ("e4_f_2_4", 300.0),   # B → D: 30초
    ("e5_f_4_3", 300.0),   # D → C: 30초
    ("e6_f_5_6", 300.0),   # E → F: 30초
], columns=["edge_id", "length"])
toy_edges = toy_edges.assign(highway="residential", free_flow_speed_kmh=36.0)

toy = RoadGraph.from_frames(toy_nodes, toy_edges, modes=("drive",))
toy

노드 6개와 방향이 있는 엣지 6개가 만들어집니다. A에서 E로는 이어지지 않습니다.

먼저 `.py`의 `trace`를 작성합니다. 아래의 직전 노드 기록은 C ← D ← B ← A를 뜻합니다.
출발점부터 나열한 목록을 반환하는지 확인합니다.

In [ ]:
prev = {"n2": "n1", "n4": "n2", "n3": "n4"}
try:
    expect("경로 복원", sol.trace(prev, "n1", "n3"), ["n1", "n2", "n4", "n3"])
    expect("같은 노드의 경로", sol.trace({}, "n1", "n1"), ["n1"])
except NotImplementedError as exc:
    print("[ ]", exc)

두 항목이 `[v]`이면 `.py`의 `dijkstra`를 작성합니다. 함수의 설명에 구현 순서가 있습니다.
힙에는 `(잠정 시간, 노드)`를 넣습니다. 도착점을 힙에서 꺼내 확정한 뒤 `trace`를 호출합니다.

교재의 예제에서 정답은 A → B → D → C, 120초입니다.

In [ ]:
try:
    seconds, path, settled = sol.dijkstra(toy, "n1", "n3")
    expect("A → C 통행시간(초)", seconds, 120.0, tol=1e-6)
    expect("A → C 경로", path, ["n1", "n2", "n4", "n3"])
    expect("확정 노드 수", settled, 4)
    expect("A → A", sol.dijkstra(toy, "n1", "n1"), (0.0, ["n1"], 1))
except NotImplementedError as exc:
    print("[ ]", exc)

도착점을 처음 발견했을 때 끝냈다면 200초가 나옵니다. 교재 3.2절의 갱신 순서를 다시 확인합니다.

경로가 없는 경우와 그래프에 없는 노드를 요청한 경우에는 `.py`에서 불러온 `NoPath`를 발생시킵니다.
아래 셀은 `NoPath`만 정상 처리로 인정합니다. 변수명 오류 같은 다른 예외는 그대로 드러납니다.

In [ ]:
for source, target in [("n1", "n5"), ("n3", "n1"), ("n_없는노드", "n3")]:
    try:
        sol.dijkstra(toy, source, target)
    except NotImplementedError as exc:
        print("[ ]", exc)
        break
    except sol.NoPath:
        print(f"[v] {source} → {target}: 경로 없음")
    else:
        print(f"[x] {source} → {target}: NoPath를 발생시켜야 합니다")

A → E는 연결이 끊어져 있고, C → A는 역방향 엣지가 없어 갈 수 없습니다.
세 번째 요청은 출발 노드가 그래프에 없습니다.

## 3. 하남시에서 실행하고 경로 검증하기 (교재 3.4, 3.6)

앞에서 읽은 `drive`, `start`, `goal`에 같은 함수를 적용합니다.

In [ ]:
try:
    seconds, path, settled = sol.dijkstra(drive, start, goal)
    print(f"통행시간: {seconds:.2f}초 ({seconds / 60:.2f}분)")
    print(f"경로에 포함된 노드: {len(path):,}개")
    print(f"탐색 중 확정한 노드: {settled:,}개")
except NotImplementedError as exc:
    print("[ ]", exc)

최소 통행시간은 약 332.42초입니다. 제공된 구현에서는 경로 노드 83개, 확정 노드 4,565개가 나옵니다.
동점 처리 방식에 따라 경로나 확정 노드 수가 달라도 최소 통행시간은 같아야 합니다.

반환된 경로의 양 끝과 연결을 확인하고, 엣지 비용을 다시 더합니다.
같은 두 노드 사이에 엣지가 여럿이면 통행시간이 가장 작은 값을 씁니다.

In [ ]:
try:
    seconds, path, settled = sol.dijkstra(drive, start, goal)
    assert path[0] == start and path[-1] == goal, "경로의 출발·도착 노드를 확인합니다."
    total = 0.0
    for u, v in zip(path, path[1:]):
        costs = [cost for neighbor, cost, _ in drive.neighbors(u) if neighbor == v]
        assert costs, f"{u} → {v} 엣지가 없습니다."
        total += min(costs)
    expect("경로의 엣지 비용 합(초)", total, seconds, tol=1e-6)
except NotImplementedError as exc:
    print("[ ]", exc)

시간과 경로가 함께 맞아야 합니다. 노드 좌표 사이의 직선거리를 더하는 계산은 이 비용 검증을 대신하지 못합니다.

## 4. NetworkX와 최소 통행시간 비교하기 (교재 3.4)

같은 그래프를 `NetworkX`의 방향 그래프로 옮깁니다.
평행 엣지가 있으면 통행시간이 가장 작은 것만 남깁니다. `weight`의 단위도 초입니다.

In [ ]:
import random
import networkx as nx

nxg = nx.DiGraph()
nxg.add_nodes_from(drive.nodes)
for u in drive.nodes:
    for v, seconds_, _ in drive.neighbors(u):
        if not nxg.has_edge(u, v) or seconds_ < nxg[u][v]["weight"]:
            nxg.add_edge(u, v, weight=seconds_)

print(f"NetworkX: 노드 {nxg.number_of_nodes():,}개, 엣지 {nxg.number_of_edges():,}개")

노드 수는 같고, 엣지는 28,109개로 줄어듭니다. 같은 방향의 노드 쌍을 잇는 엣지를 하나로 합친 결과입니다.
자가 채점과 같은 방식으로 출발점과 도착점이 다르고, 경로가 있는 30쌍을 고릅니다.

In [ ]:
rng = random.Random(42)
node_list = [node for node in drive.nodes if drive.neighbors(node)]
pairs = []
while len(pairs) < 30:
    source, target = rng.choice(node_list), rng.choice(node_list)
    if source != target and nx.has_path(nxg, source, target):
        pairs.append((source, target))

print(f"대조할 노드 쌍: {len(pairs)}개")

선택한 30쌍에서 최소 통행시간을 비교합니다. 경로가 여러 개일 수 있으므로 노드 목록의 일치를 요구하지 않습니다.

In [ ]:
try:
    errors = []
    for source, target in pairs:
        want = nx.shortest_path_length(nxg, source, target, weight="weight")
        got, _, _ = sol.dijkstra(drive, source, target)
        errors.append(abs(got - want))
    expect("30쌍 최대 오차(초)", max(errors), 0.0, tol=1e-6)
except NotImplementedError as exc:
    print("[ ]", exc)

최대 오차가 허용 범위 안에 들면 선택한 30쌍의 통행시간이 일치한 것입니다.

## 5. 기본 실습 자가 채점

아래 셀은 `.py`에 저장된 `dijkstra`를 검사합니다. 다섯 항목이 모두 `PASS`이면 기본 실습을 마칩니다.
`trace`는 다익스트라의 반환 경로로 함께 검증되며, 미구현 상태에서는 `FAIL`이 나오는 것이 정상입니다.
터미널에서는 `python labs/check.py ch03`으로 같은 검사를 실행합니다.

In [ ]:
from check import check

report = check("ch03")

제출할 것은 작성한 `labs/ch03_dijkstra.py`, 위 채점 출력, 막혔던 지점과 해결 방법 3~5줄입니다.
A*를 작성하지 않아도 기본 채점에는 영향이 없습니다.

## 6. 추가 실습: A*와 탐색 비용 비교하기 (교재 3.5~3.6)

`.py`의 `astar`를 작성합니다. 힙에는 `(누적 시간 + h, 누적 시간, 노드)`를 넣고,
이웃을 갱신할 때는 `h`가 포함되지 않은 누적 시간을 사용합니다.

먼저 이번 목적지에 대한 휴리스틱이 교재 3.5절의 일관성 조건을 만족하는지 확인합니다.

In [ ]:
from smartmob.teaching.graph import haversine_km

vmax = drive.max_speed_kmh()
h = {
    node: haversine_km(*drive.coord[node], *drive.coord[goal]) / vmax * 3600
    for node in drive.nodes
}
violations = sum(
    h[u] > seconds_ + h[v] + 1e-9
    for u in drive.nodes
    for v, seconds_, _ in drive.neighbors(u)
)
expect("도착점의 h", h[goal], 0.0)
expect("일관성 조건 위반 엣지", violations, 0)

이 목적지에서는 위반 엣지가 없습니다. A*의 최소 통행시간을 다익스트라와 비교합니다.
노드 하나까지의 잠정 시간이 갱신될 때마다 `h`가 후보의 우선순위에 들어갑니다.

In [ ]:
try:
    d_sec, d_path, d_settled = sol.dijkstra(drive, start, goal)
    a_sec, a_path, a_settled = sol.astar(drive, start, goal)
    expect("A* 통행시간(초)", a_sec, d_sec, tol=1e-6)
    print(f"확정 노드: 다익스트라 {d_settled:,}개 / A* {a_settled:,}개")
except NotImplementedError as exc:
    print("[ ]", exc)

제공된 구현에서는 같은 통행시간을 구하면서 확정 노드가 4,565개에서 3,571개로 줄어듭니다.
실행시간도 줄었는지는 같은 질의를 각각 다섯 번 실행해 확인합니다.

In [ ]:
import time

for name, search in [("다익스트라", sol.dijkstra), ("A*", sol.astar)]:
    try:
        started = time.perf_counter()
        for _ in range(5):
            search(drive, start, goal)
        mean_ms = (time.perf_counter() - started) / 5 * 1000
        print(f"{name}: 5회 평균 {mean_ms:.2f}ms")
    except NotImplementedError as exc:
        print("[ ]", exc)

휴리스틱 계산에도 시간이 들므로, 탐색량과 실행시간의 감소 여부를 각각 읽습니다.
위 값은 이 출발·도착 쌍의 측정 결과입니다. 다른 쌍이나 컴퓨터에서는 달라질 수 있습니다.

## 7. 4장에서 사용할 함수 확인하기 (교재 3.8)

4장부터는 `smartmob.teaching.dijkstra`에 제공된 함수를 사용합니다.
`Path` 객체의 `duration_s`가 통행시간(초), `nodes`가 경로 노드 목록, `settled`가 확정 노드 수입니다.

In [ ]:
from smartmob.teaching.dijkstra import dijkstra as lib_dijkstra

p = lib_dijkstra(drive, start, goal)
print(f"제공된 함수: {p.duration_s:.2f}초, 경로 노드 {len(p.nodes)}개")
try:
    seconds, path, settled = sol.dijkstra(drive, start, goal)
    expect("내 구현의 통행시간(초)", seconds, p.duration_s, tol=1e-6)
except NotImplementedError as exc:
    print("[ ]", exc)

두 함수는 결과를 꺼내는 형식이 다르며, 같은 그래프에서는 최소 통행시간이 같아야 합니다.
4장에서는 도로의 연결을 유지한 채 시간대별 속도로 엣지 비용을 바꾸고, 경로를 다시 찾습니다.